In [1]:
import numpy as np
from gensim.models import Word2Vec
import gensim.downloader as api
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import torch

/Users/harshitgoswami/Desktop/GENAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs = [
    "Artificial intelligence is changing technology",
    "Machine learning is a part of AI",
    "Deep learning uses neural networks",
    "Natural language processing deals with human language",
    "Python is widely used for ML"
]

q = "artificial intelligence and machine learning"

In [9]:
sentences = [d.lower().split() for d in docs]
w2v = Word2Vec(sentences, vector_size=50, min_count=1)

def avg(model, text):
    words = text.lower().split()
    return np.mean([model.wv[w] for w in words if w in model.wv], axis=0)


In [10]:
# GloVe
glove = api.load("glove-wiki-gigaword-50")
def glove_vec(text):
    words = text.lower().split()
    return np.mean([glove[w] for w in words if w in glove], axis=0)


[==================================================] 100.0% 66.0/66.0MB downloaded


In [11]:
# BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def bert_vec(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, padding=True)
    with torch.no_grad():
        output = bert(**inputs)
    return output.last_hidden_state.mean(dim=1)[0].numpy()


In [12]:
# Retrieve top 5 documents
def top5(doc_vectors, query_vector):
    scores = cosine_similarity(
        [query_vector], doc_vectors
    )[0]
    ids = np.argsort(scores)[::-1][:5]
    for i in ids:
        print(docs[i], round(scores[i], 3))

In [13]:
print("WORD2VEC")
top5([avg(w2v, d) for d in docs], avg(w2v, q))

print("\nGLOVE")
top5([glove_vec(d) for d in docs], glove_vec(q))

print("\nBERT")
top5([bert_vec(d) for d in docs], bert_vec(q))

WORD2VEC
Machine learning is a part of AI 0.36
Artificial intelligence is changing technology 0.349
Deep learning uses neural networks 0.158
Natural language processing deals with human language -0.082
Python is widely used for ML -0.109

GLOVE
Artificial intelligence is changing technology 0.935
Machine learning is a part of AI 0.901
Deep learning uses neural networks 0.883
Natural language processing deals with human language 0.865
Python is widely used for ML 0.835

BERT
Machine learning is a part of AI 0.815
Deep learning uses neural networks 0.803
Natural language processing deals with human language 0.752
Artificial intelligence is changing technology 0.739
Python is widely used for ML 0.536
